# Regenerate s1K reasoning and answers with DeepSeek Pro

Run on **Colab CPU**. Add `HF_TOKEN` (read access) and `DEEPSEEK_API_KEY` to Colab Secrets
and enable notebook access. This notebook contains its helper code: no Git push or GPU
is required. Use a fresh runtime when opening a new version of this notebook.

Source: `Seungjun/dp_removed_s1K-1.1`, pinned to the same **996 rows** used by the project
before the SFT length filter. Every original column and row is preserved in the full export.

DeepSeek Pro solves the original **question only**; it is not given the old reasoning,
answer, reference solution, or grade. Thinking is enabled. Its native `reasoning_content`
becomes `deepseek-v4-pro_reasoning`; its final `content` becomes `deepseek-v4-pro_answer`.
The final response must contain four explicit sections in order: **Planning**,
**Solution and Evaluation**, **Reflection**, and **Exploration**, followed by
**Final answer:**. Planning explains the strategy choice. Solution and Evaluation
contains the full derivation with concrete intermediate checks. Reflection develops a
concrete refinement and explains what it improves, or explains the specific tradeoff
for retaining the original approach. It does not require an error. Exploration develops a
concrete alternative and compares it with the main method even when that method succeeded;
a full second solution is not required. Instructive errors or dead ends from API thinking
may be included with their detection and correction, but must not be invented.
No word or token length target is requested.
Here "answer" means the API's final `content` field, not a literal `<answer>` wrapper.
There is **no answer-correctness filter**.

**The intended new training target is `deepseek-v4-pro_answer`: the complete structured
worked solution.** `deepseek-v4-pro_reasoning` retains raw API thinking for inspection.
This notebook does not change the existing SFT loader to use either new column.

**Updated format:** reopen this notebook in a fresh Colab runtime and rerun the smoke test.
This replaces the natural-reasoning format with required sections and concrete content
requirements. It is our adaptation of Kimi's cognitive processes, not a published Kimi
prompt or a claim that the paper prescribes four fixed sequential stages.
The new prompt/code creates a fresh output folder, leaving previous results intact.
Seed 42 selects the same 20 questions, so you can compare the old and updated smoke runs.

1. Run setup, settings, and source-loading cells.
2. Run **Smoke test** for 20 reproducibly random examples and download the five-column CSV.
3. Use the separate **Full run** cell to generate the entire dataset, reusing completed rows.

A default **Run all** runs the paid smoke test only; `RUN_FULL_GENERATION` starts false.
Both modes save each completed response immediately to Google Drive. Rerun a cell after
interruption to resume; unsuccessful rows are attempted again, completed rows are reused.
Use only **one notebook/runtime at a time** for the same output folder.

In [ ]:
PROJECT_ROOT = "/content/drive/MyDrive/LG-AIME-S1-DeepSeek"

In [ ]:
# @title Install CPU dependencies, mount Drive, and read secrets
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "datasets==5.0.1", "huggingface-hub==0.36.2", "httpx==0.28.1",
    "PyYAML==6.0.3", "tqdm==4.67.1"])
from google.colab import drive, userdata, files
drive.mount("/content/drive")
HF_TOKEN = userdata.get("HF_TOKEN")
DEEPSEEK_API_KEY = userdata.get("DEEPSEEK_API_KEY")
if not HF_TOKEN or not DEEPSEEK_API_KEY:
    raise ValueError("Enable notebook access to HF_TOKEN and DEEPSEEK_API_KEY in Secrets")

In [ ]:
# @title Install the bundled, tested project code (self-contained notebook)
import base64, io, sys, zipfile
from pathlib import Path
CODE_ROOT = Path("/content/lg-s1-regeneration")
CODE_ROOT.mkdir(parents=True, exist_ok=True)
BUNDLE = 'UEsDBBQAAAAIAAAAIVwAAAAAAgAAAAAAAAASAAAAY29tbW9uL19faW5pdF9fLnB5AwBQSwMEFAAAAAgAAAAhXHQtxyXcAgAAhggAAAwAAABjb21tb24vaW8ucHnVVUtv2zAMvudXEDrJa2psBbZDgOyybodh6AbscemKQI7pRY0jGZLcJCj630dKTuLEwbDDephOJmk+vk8kJYS4bp0qaoSPXz/fQKVr9KBMCapp0JSX1tRbuLetM6r2sNZhAQ7n9gGTkzYBnWubgCUEpWufCyFGI71qrAuwUH5R62In3ntrdt/WjypnV9CowL9Ap/5C4mg0KrGCUv9CH+SDqlvM4PIt+OAmI6DjMFA9u+i5X6ir129kNPHhNHnZrhqfnMfgKfRsiVs//eZYRuNbhzPl51pPPxAwzHI0c1uizGKYLF/gpisg6+pZOx1wxsElFz2GVFkqiTUwjeVHa7bX5o1yaEK+WpbaySTs69hoH2Z2GcXkEpCJUG5L0aI7Uz7zbVXpTYycp2+4AJGHVSOSW7yYvW9u6e6kWIsMlIdqMqRmx0x1jowxXWtJZU6vsr1nlUf8Uvw0oq+t6tYv5EFjfV75rZlLMlEvGSuzE2C5w6ZWc+x4OiW37th1du3/L3Ir67hqIi8WfzD0+es1J/10thU5+/OwnGb6lOZ/zXKkK/olptSQqecnIwF2qI7hviDE2CjtZrytuhx/wq8rMDYkOBGrl9kBSLeJbu/SXqJbpwidtKJ1QpJwF4XgML28gJSWLIU4Rxg7njK2XhAyYI6P28oHRVtzSqwErOseKXxqbTDamAYWTuwdNrYcR+VTkNPySBvcdvgbN/w07ZXaqtJLjnacBjdzbAKk2+ZX5hp51b53zroxfDeapZ4uG2Y5oY/fJ648IZOvsr2Gs9MqLz1zKovYOsNwfIgx15q5ojaMJGZn/xqyEEErTff3g1dorFhW4p2Nz2B8RT+BClBsA8JjDP00gUe+3icahPjo3ViDR0G5c/I0nTwJg2s6B/7voFa5R1zKl2O4GiLczWFxGLGuo7mgboZqoohGvNjOdJkG4+gVJoiOa74VuhR32SR2RH8Vno5gf/7i0nga/QZQSwMEFAAAAAgAAAAhXAAAAAACAAAAAAAAABQAAABwaXBlbGluZS9fX2luaXRfXy5weQMAUEsDBBQAAAAIAAAAIVwJkiYk2hQAAEVGAAAZAAAAcGlwZWxpbmUvcmVnZW5lcmF0ZV9zMS5web08a3PktpHf9SsQ7FWZo6W4ku244tnQW85aTrac2Fu7e7m6Gs+xKA5GgsUhGT70sKL/ft2NBwESo5F9VacP3iEBNBr97kbTnPO3dVUMbSuqnn0nRPNRiGvWiryrK1ldsktRiTbvZV29ZlXN8qq7Fe1JUcOCoq9E17GtLHvRwtyEc350JHdN3fYs7+6rQtbmsaibe/u7uzE/27za1Dv7JMyvXu7E0batd2yT9wKfmB4xzzHN+bWu9Dyxy2WZDL0sOzO1ydtO4PysrzOzTs1u8v6qlBdm5nt4tJhf9X1zd6TmFfVuV1dmGhzHeZvI2gzkTSOqTfYL0KyM2UZeiq6PkYj23W0rAQ98cH+XR0dHG7Glo3R9vmuixfKIwV8r+qGt7GGTqr6NzHnhkMUikV29rdtd3kcLDeQmLyWdtquHthBRW992MeBabeWlBts1omCpfrfiaiJf05jcslJUtGrB/pDS3BUXd/BPLzYZvudrVrf6eNOJALMHEcrUKF+rDeksuewE+2deDuK8bes24h9pXwC03YoWBAhp2l8J1siqEhu2EQgs38kKTrNh3dkPSIi8Ez1fEFQ4OIPtmazwn27cCgeqHIWlsocs6nLYVYB7coModIbE5g/OXdXA205WwIJKES65FH2EkBYx6/p2gefGWTC0wtfrBN7KZgoreNwt/4fsOtQlRW/Wi7t+yR4QziM33CtrEBbNOYU6SHh9LSq9hVEGJIOVcFqkXx4d5LCeCMPuukjxrxVNzdcxkS+1LAUg+LJrStnrt/QbXs4O7tJA3AA96yo1sNUjgqIzpepkStSBg4iSBJnSKKmBJ+XZ1RIc0FSEN+19lm/BIGUditGmi4jtmoqa2fTKkVAF5zQ5pVcAYja2y+8iGI/ZFmjXa5gKC3FXiKZ3OD4u9iCFoAVNlIaeOFaBnZCNoFeR3tfZOxo3j9mn+8b8/OlGtIDwLT0ugqjgoY+OijIHS/49kL/89v07Jbcfhgq3c9eCgf92AE2FgYJ8QswuZFmibIOCyIpYBpD/BXrWM4ErO7Yb4HfX1w2cFkZv6/ZaeYqjtz/99cd3n9798zz7eP7207uffvwIgrDi78u8Qt/DYwamohxwI3A8G3aOhKF9ceiD2JZgm/TT+V1T1spRgbgraVDOKlN2MgMlRPVXL2NWD30z9HpwPN/bK1Fcs6YVHZySwDGw0/fk/IBKcsfqLXPdHxw8NzRhnSi3xjnCUjpn2MgYNFzrot7NTYtm1YrvlCHJtmAaywxwbOqqswaczocUVM+7vL0WLTy3IhFdkTci8s684gqMppKaztdaqKsNbGQVE0DA5A34rTZqefRG7hb/s2I/9+vj6M3y5+OfjxdvOHtptnzJcI5+r6YBi9Q+Wni1t9HbkB85Gw+sjpIotxrZY9dtthnA/BSoMCHkNe5l5yi3Jr2auFrqHVenaMLzFtznHlu+BwcUX4GWWsklX+zZyNkH1y+Wv20bdTowYEU5dOM+nZIr5MuEl2ZEC8MVxB77GDjK1ZSTLx7O4q8e6fHl4o1lIfxYnZ18uV4li7Wai+/e/1nv+Q23EF8y/m+e/FLLKhqlTk9bkHvWD+ihDcoLZznwUO2wfDORoJ/bN/8xlSJ0ViTx+riWDODtC/Ak27oE0tO+egbua2iz1uZbRTgIRq+erg3gvLaBiH4T28DjV2CymRZb8AtfIhHxpKjBxKowY6IAc8nYhrVAb7R80D8eudWvDiIEsSHwoGBpal5YjMimKhLC7ub1Xi00Z8pA8rJ6C3hsrMohIWTsEllUww4TBxEZcjsUeME+oj/Iewr7KoiHyGnIFthgYABEUgJtVMBSYvZRwwJL8886ByJGVSyCVQhyVJwFK8CbgF52YMy77T3AgFyh6e8NkMTCuAB2bPJWKpFC86SFbe2yToKUnrE/k/kKHM0HZIhnJq5otTU9i8DeCZwDlygDYmYShdUrRV2l3YDPZN43hoDzPXwDZaYp6wTBdjTi8DybuOVEyFEGjRZ+ZvizdsTRM/cgjWckfnl1H105yM/Ns9JeV28Xh0RUB4Cuh+BexKiW2bjngxizXIh6FHgMILJMVrLPsgj9eszcCNQGEG1d9w5GODPRMS0mvckGcmr8oRImf54CNZvpxrj4N42E7R6xC2ac77kGgO5MWnFvkHuiHWE64MdFaBkeeNfflwKjLOtl4HfIAT/OsgJg3tRTEbA1Qobf7VAASzDBNCUH/hwY1tshmHkcGQIxCcCeEwup8GwfsOdAMHo0Db5nOfG5cUMhkrBb2V+BIgwtSMoliKS8EWMsQHqkjOWJ2p558ZDm7nPQVUSqfIlpIPVsvJQ+fITv5xio4P8C/SLY4w0cD4BTqq9gegiiLTCu9ik8pj55zv6D1H47peIYMD0b47yRU92CV75GwQuI2uuNKEfaPnBU9E6I65ObL08ALCqTfbUFi3TFHw8d4D+rbmgacuhj1U5ttJghYOUoE1ugXe/iAtkh7n8lL6/wX8hMn7E5POdsRKC/ktU1iqkGP3M5bt6DCBWm5Fjco5LJqqccaD4GfvbsEDbOAsu8nDV1pxgMwMG8t4eRgqNnWD8A/zdDyhsDpE4PIeUscJCq6qoSl/nTeEFwkY67UqWENk3ZF19/8fnZV8b8nCZf/9FOhXAqa9SsgwR7p/N0Kze0ByuENJm8ghYSJMrsM6wLYBSoSyxq39ORWrLKzFQ8Zgv7uXMPk+/dpJQAG6JnmNHq1BJAVX7QXUt/M28cKWrHno8ELJ2jQNH71AJ0O6AlX2PhNN81peh8ezCROwIxiprlvgINP2xF9rm40v4Q6+LerJO/Cj4JOUZboAqjLPWNFGRwIVdMa7Wz2L9Qx1pzox7aWFVvGdnyGXA9OKn0HiTDX1UcBzZJl31ZXmIZ/h7SG0iDXwMvtwPVYvua1TeipVq8KcxupShBMCYUI5JmmDwXlBmoO4vkA/0ThbkPpnzD14tEMSKaYF1dishydhErdo+7vmA/VeU90xV1SCwFIMrKuoBkCC8uIOy7EGVN8b/yTa3ohl1+ASyXGyxE9fdjViMRg50tZqWmfr/CS48ogyi5FFm2SOi2ApOoCDCiMVknweG1o4Z6uxHsgykbL13FAM/i4wHD/ovHqZxCWA1QCRE30mavGG8HsIjww5O/VxaZ1fLsq/UcWrK73sg2anK86urST+0gYiUWWX1NjyMKu7ySWzQ8EP16zONmEzyA/gmH+31n9mvo3BM0A8t7OVlQidvMXG8ssXoRULM4oF2T4j1HG11BqJP1oFG0FlKvS9Hr6QYVD8YERJvfZsCOzIQCk6UzrPzVTj01U9eJsPD7vOzEZCJugbqV2eBjPnHMQcaLtmiUqVcY5CjuJjgEUY95nojgL2AVsACRMm913veY93a0uuRT8wpn2ZCZsLd/kQsNrwWbXLZAYllOpI7mlWC+lOCNByEzSJBtUKw38g2iA2ClJqyAuRtxx9drQglf+dtloEyUYqSKjkc2+e2GogBSi43OfgnQvCrt7Em3ZjQtBuwX9AgpXt4PYFQx5wdGowL0gjvb5DfC5NeEn7OFe686oyLNDVLf1AOCcw6Sh6bT9bWqAdzmEiKdsu4JAxc7mkOpmSYl5uhgp699ruQIwVyIJ10JYVdkb4DUSiyBmZDJ3PXsaogHQGkKyB4Xczb7a9LZGnDI89xkNctlDwRrM2roqZpjRSmFimB82TiYG13Um/u5cVW2fOkZdjCjxqjAyAPv7xsBP7io0OFt+OPUEk2TneW+JAghU6C79GLoCTwnCF/Ow/KYahgi3+0xWDswafklmfM57R94W5d0mO6+A6PClSdBr++7kjH3fZzfu45Qhg4M5xwGhhkr4lCA/1aeZvflK06cpmr6dN91yNyildLGkQI2inMC6dNLdrYIaYgS61HdJkKP5UAq008lfXKtSWYSq1ZKtZdMWySTvYD3CMpZkDQkUAgRVuVIUedSNra+ABQng+iMQGoCwCEfJ4hhAgJb5Sj45q7Z0g6vXq0Ndkdmd8hEsNH0GLuic7Lo6UQt0LCgkFNXiRjUEy+UYieQRYMJf1Vc5f0rbbt1+Q9tcopaPC+PjWxYcWymybQLUDZWbZSodxAObMSTGCUU5GPVSoOJwhti64ALn7z+fKrCKxkarKdGT+6cyU2KUJUbkxu+iMcxEhpnWJdewkKktBvr0ZDwNy1YWWflfBA3GtBqOLPoOQR/fsTiqoZYESNyWA36TI+oz6fr2VxtnrACTdNWxmDx+VxrQM0lHKzSsxWObpCnDBAcxJth38+Aw/Fld2XxUNPVS52OBhaZaAKWhVlpoFKluW64uvGCMG6CN+SEeRnYAP9widscNFKBru5x+KKuy3HAlnyfA85rA7Cw/EaAOaAA/dweB9TiUOvDHoQ0Bk9cEvxOPdIESWeSk5pNPQ6n6imsQ5pJaZhz4TXe6VPvaY+ikoVJQdtVOcUQAAsZPoWxx8AJZ8O8lttRRNUCWdklz1Fl8hmU16dGlqloC+IzbLeyQBudaQMCpkk1e6FbulC5ke+BdLMQ9Tcmf/v06f1HOi6VTsCN4PjcOyiSMLwfLpJnWW8Xab0akf7y9E8x+/L0a/jP518/UtlHDX6Tsj+ens4lWntEarszIGezXrDvappS1pe2Woi1fNHiVV0rKEcHpxpTdbK9kJDjtnj522JRhVqUMBjFi9cDQq6Fg3JQWgeUpn/TLUdqsgc14zEgDb7zD3WpeeTVBzBCDrNPaDYP2AK86dc8SpG0YQf/3FzFY30ozcDEZT4Qh9IPnLoX+IQq8ZNV3MmxTdebkuRPEGh2WAww/W8q5lF9a3vl2pVSzMJ/N/MxH0HuLZIsw2aKLAsj67bo/SDu9a93GJvOG/e+7cH0Xwy9+P86Bf9HXqKFE5sPWgiX2NB16HAmzAMWNh2WZWzdHaQc2wmimVycmEg+Zl8E0loqB0wTd/xDE4wmYV/Hrd+8uOX2dkPppsb0Mycg/Wz9+BriDez38wzF/AI69Mcv8hJdOHltox+d6Hu8RYyxLAspkGiHKmFvtc3fqGZXPOIm4bPjjcykG12dSKRp4ApqHxHcns1oJEGPKoLO4kQZPHMzJe6u8gEzotfWUWkcn0UDnrdCnwYEGXt9e+ZXxzRxgApYcKcytQic3LPxQSkf2sp7fZEX1/V2iwZJVtG+G56nrQo7Zp+z42ODso/VC/Y3ENqhZR/I/H6L5peJG2DqLXIWUiRUbLHpGE4Sd6AzSoBODGr6Ps13K/tqQXpR7NpEsHqT6otuUxFjRY6qwnjK7FqAET2O8Z76EghtCty9sY7pj4Chk/5xzj8QP1Shr+tE91pfdW1zWRpBwNZbhjUz6u+tVdtvwv6Cyq0aep37V9vvqvnqVRhDdze+vH66kh1DWZHjvU1e4G1poM1SHfnQhdB35+fvP56f/5CBZch+OP9vBG1azVyg9naHuiX1o7cpXth0Yhyj1hh8aV5QgFHdRxIvOKkfGkMbtUxXYVQzk8R4yKx67sUefnhhuv7MfZW5KphUOJ2SrucT+ry7dvqDlRphFoeSMc6a1hvU5xn/2uySfOjtJy/44sgX7UM1PwdBE3xgpqJ14e/wOC35hAKQaeWk0V1xKaPmVjkjsZEY5V5sUVtOggpFCjxW1NfgSlKXt6hrZD1S0JhoBmnO1r2fR2xEV6SjZR4bz8AVD7BHam6OOX5I1KFBTBF7o9Y+0k5Yp4Khb/HFWyrezGOvi7wT2dCWKVlF84RmUgec6QP199et/NXcWG35XwSY+ZY9aI17nFZcSWhUzJU+XXxyrZH9NQdWyp3su1Qd6O/0gDYSqwqV7nlKA20kz/LaCOcaiA86dSMOQJwwkaIwVRfbV5BDG00N4m3oyyD8I0HBmA+FRYvu/gjcrYuawrtfct+7FMTZRH1nT88Co5ZhnW8r7yITBfyfxXz2B6k6iO9VSgXOxdFcgoxxMvYAQjL6LgbeR4amCoNsrC4Ht7cRwaTNyFXoxWJe4ApWWvHPd9qXOfYgR8eEcbCSlZflHkiIPq6ji3tcv5/3OJwUGGGWe0qeT6AV65gpU9kHCbh/1xhEEywlmkHwkWhy0C3M8UORKcq6E0Fjvfc2ESOktp8GLlg29WOS8zv1RWVZMluv1423ZOaqAUZUeaojcmKwMoBoUajZgWa35CFVSozdC900KsFNxwY41bOB/awA+TmNd9dVfVvp8xCs3xZH0O5UkaSNVUFQL/tDGrj0P4TRR2r5Mfhgr9kAM7AtBDQaiUeNKroniIdxobNPUVEfjthulTGKOdzN+j1yyUWJPqhqbwSG0Fh+UQGMbu3RT+4HBUoDsAlhcv2+c0tOasZvaexQciEmYVA+XGJThvoIZHztGeswN8xt0/5bcG++U7GeWte5ObdYmfvsh5kyHh9Pb/jmFjnYgLG0qfvY7LWeFyzRBOyB6HWDjOB0C9izYE2IY3hjj6s/X82mF3n6Psls6vUZwLB2q/uqwvZvcr/gg5vcPRwCpaopPgj17jAWbmF5ioU3FoNwHgTnXF9NgLkjzwdj7kqDkPS1l8PIFj/E1y4Ag4hg5D0PJEYFHXY7LNDiBa51ALC9H42PlXd/hJ2MCARPyO0wHclMZY6o6e6rYMuVgbK/YWksucOpzYdlbt5oxwd987ZqvSs+zPQeHlXLz7Tbh6pEtmdFj9Im6ykB1RfdWGfTvQqZamhHEzBexU87MLDGtyS+DfYT+JidElKnhNOAuxKqvvK6n96bdgXTJTHyCz/OGV9SnmVbKSy4YAOXdgavgFPqgKZ5Sz8uAotKdxWZB92zFVtb85RrXk5Md9lN81vbKzEjIjkYXOJ2UcRMvYH86lJ/wWtt72zMGNI9ivr8Zj8/W4YoEdNcSxdkTN7KDr967G78sh+WxepWaSMuTDACy+jq6S7iOD/pd82knEdRml2Z1GCNI47fGEAYXqP4pXzotyd/OukkfuFdidtSViLlnPKr7TzgJH5ibo77fSeL/r/oRbSNVcMukTul/6Lbh8SyyymnS7m8rOo2dDunYCb0j8p9AzG2Owk9bGQd8iJMJuBHU+aFiJBYzkf63vfIIS5EXP+fDxLgBZYZtJyqqrAxqOoK0cxUMya0d0Qf94hZAGXdt/fAcQaYLTXRqBVfGo2yjjbTE/co0+PR/wJQSwMEFAAAAAgAAAAhXNrROUVPCAAA0REAACMAAABjb25maWdzL3JlZ2VuZXJhdGVfczFfZGVlcHNlZWsueWFtbI1YUW/juBF+z68gso9NvLaTOBvf02K7Bxyu7RW3WxQHFDAoaWRxQ5E6krLXuPa/95shZcvdO6BPsURyOPPNN9+MEv0YatreKBVo8Fv1iUa3/zK6t82wC9T7AzW7uPrxfrVYyZ6Dica7rdo0q+fnqt0s9erlqakf6nX1+NDQRr+jtl2tH5dPelW/PK5xqPauNfutaqjVo014Ewdr0laloI3DI30dqE64KPhj3KqXl00+lcilXWP2FLF58/z0uG6fqH1Yty/03D48Py31Q/WCq+jpcbl5bp6pfl4uH1ebtaZ3zYOu18v1y+Zl/aLbzcPTTe3t2LvIof46wqSEMf3CSx/M3jhtEbaO3hknLtMQiV53qTPuFa928PkLnPXhND+iXTxSmO3XKVE/pBs9GL6w0pF2Y7Bb1aU0xO3bt1hYTLsXte+xqfcN2YuN+8Pj/RC8oF4c2gFbHwBGZ/YdFpIfdsNWrRZLPLxRPw0UNEejrarJWJxQ2H85r/6ksqd36m8/fVYaKwIANSrpsKekLLl96hZi7n15uremN7wnUBy8i6RMVK3V+z01d8rRgQJSSZq36AirCGewlAiJ1fyLzfX66y75V0IC1OphtXwuzKjHEMjVJ7zlrPfG7YpTOwMChAPQjYSNDQ4uF2uBI68n05Mf02V59W65LFcFSsEQ3j3IgRROMGeSmVtbn5f4xPn1ZnkTe3i6FWZKALyZLSMtzVaB1Lh2wM0At9eJN8Z0srTFnzDWaQzM5Ql0OVdzWoR7St2rv1vtyhI/fgIzeR3JadRHRDzqwkle/Zlam8+XFx+/DtaHaUs7IyACCa/Mw9vv+W1J9vb2BjwCG7fq3/c4gesOpFJHKo4DKhFpw3plqVfGqY9ub03sFuoXKEO2fkn80VirKlJjZMp4lUjXndIcYq+tBRGExKrzR17OGOCq4Md9N90SlTWv7ICJC/UZbmBbg6Ni3Dt7wgvQ90DM8I65duUEKOeTOrF3kQYNIMSWlOfbC9XrDhiTvVPRS6zTcRg1KZJtVT/GJDIDFZIt7Wituqr+YA6Cc7xTr3RStbb1aKc3yBaMfYEV05o6v12oP3vxD2pqdS0w94gsASXA3SNBJ+XbWUmiZplGwJNspGNHAfUCu/+IUj91Ah4wgqc2hyxUUB0ww/EIwHALp+FOSRjAy4eGS7z11vojDFdiYeLKfWYFkuBoAboRPJqsXkFS0nUfoc4cIQqLo0GgNQtM1mfx9c2bOaV/aPDetHKp2nvhYSMPjOEeiXWqRWBR3sNOlEaQAN6nlLNJXEn4uT8VlNEiLPt07E5IHwLBaYm1OPmdajLuPfCzfMr50Qn+WsiS2RWF+ubs9R9X3gcdOFNjujAjls0MdG1Hhl8WKUYOGHFeEeb/IcuHjupX4PoFKi1yBx4YhgBsRasEQvn+wWNVCTn4GfEFE7kMM+IaAQxAYghyOI4VrknF2QpANEI7sIdTycnTXEfR7LEetNvjYYQ8wk9s0hy/cQ0N5DiV80imFB0ZVsBfUa41hMHNIErhw2F5I1FPqYPNfCgBsKQrFhkCBO+Raljs+aI4sppEdSupFRuRc3CrtPVO+o5xcWyBoBHugWwtNxyBEcMJacusUhSCRwVMrMk8Q6Op5f6KUHGMsT2x96JQOhXMJ2pcye7P3ByiYZXi6ARsVj7RQV1hSlpcaK/VuWLQWgegEFH5g2RDhgaQNHUeLnXeTBSt/WgbllbTwzamLq7m2iLP6YRASsg1fu/Joc1bfp+tTe3WgYji/ccSNrsqvVtnJuDZgaMHsn6QNu3qQEI2CINkAOktGZyGPY4v7EdevKL0nXAoT3J338T0R5kvwUXxWaNNZmlkR6dZ6gzuQv011zKkFMNFZrpgJfowVS44gbo1zbfkwPbPv2cYIwsuZrEA3Mfg3Z5ZeAEB9QhFhKtWsgkzl7pGuchQzCQb0V9qYM61NUuxQuNpLHZcKo+hKiIG56DogtjUImAKN09+C2fEeVCbhWtiHmdi7mU1CqS+bSlEhIMMaFZgaNWEs2Rtpp4C3MRNCGxDOC2pYl3ly8CHaeO3CYGl0t2MO7ALUmbcb3jMRCRzRLj/YnKfyul6aPkwVdOMhdpC/xx2HGYFRtwsTCsO9VLK0wq0okYeqRG3Cqm5LXCTwYQ4sVTay1jnJ+bpvIn5KvKIKUQiJ0MKxhYkE0HkQYEvRNJ0ZaTicg0V9jL8Z/6Kd7kIFucK5GIySbTHVxw+RSHxJPDsUGlpoC8GzYi4DKsjMSfPg3QeTq9In4eMX0cDYnCOmQ5CWnBXi67dauzp6Fya3C19jAbGb7+plimzkBzTS5nlvjLi282xVDdXCZq1B1yIGQKUopjFUcYLyfsvV0PaZebhAus0zJRRQ74tkI4W8s2tXwduRYVdUgMaqRGdYa8KBSiv6QoERncA7P/k5HCj8G2ZmXKYmQAHmjo3Fd5xD4XvsXwAtpyH3+m4XGW1fKhg+j9OAwV/YKDlcYr5EeBB+PKcU4aSqetPW1rzlXUdYP9Q3JBpt8TJZdhgqgMRm5iFDkPSiLH6dL6emwJPvvOPwnPqWl0FHi04BadFkY/5199R5KX3YAVLUJmYdNPIAU7Yj/j2lAiO6DvS2T2XdCpCPUlI+ZxZzJrkWfBLkedREvaZmggUXYit8ACEZ+QT0EOzJHauginyc6gL9RdKczSxhQcksKX0J91DX1kAs745ufc7XgyUK2T2fXvEUMz3yDfo+TMXVt63sDqXpztxQ+p6mr+ZUv/zSSXj83mwLdVsRNXUvyqPRP/2H6aPuwqh4+9jbigmZtFrZ0YX6nuevngzYsLPWVjnf1Tc5XFpsopBoXxH25FpvLj5L1BLAQIUAxQAAAAIAAAAIVwAAAAAAgAAAAAAAAASAAAAAAAAAAAAAACAAQAAAABjb21tb24vX19pbml0X18ucHlQSwECFAMUAAAACAAAACFcdC3HJdwCAACGCAAADAAAAAAAAAAAAAAAgAEyAAAAY29tbW9uL2lvLnB5UEsBAhQDFAAAAAgAAAAhXAAAAAACAAAAAAAAABQAAAAAAAAAAAAAAIABOAMAAHBpcGVsaW5lL19faW5pdF9fLnB5UEsBAhQDFAAAAAgAAAAhXAmSJiTaFAAARUYAABkAAAAAAAAAAAAAAIABbAMAAHBpcGVsaW5lL3JlZ2VuZXJhdGVfczEucHlQSwECFAMUAAAACAAAACFc2tE5RU8IAADREQAAIwAAAAAAAAAAAAAAgAF9GAAAY29uZmlncy9yZWdlbmVyYXRlX3MxX2RlZXBzZWVrLnlhbWxQSwUGAAAAAAUABQBUAQAADSEAAAAA'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(BUNDLE))) as bundle:
    bundle.extractall(CODE_ROOT)
sys.path.insert(0, str(CODE_ROOT))
print("Project code ready:", CODE_ROOT)

## Settings and generation prompt

Defaults come from the bundled `configs/regenerate_s1_deepseek.yaml`, shown below.
Edit `CONFIG` in this cell before loading the source if needed. The default is
**`deepseek-v4-pro`, high thinking effort, 16 concurrent requests** with request pacing
and bounded retries for rate limits and transient API failures. Temperature is omitted
because DeepSeek ignores it in thinking mode. The sampling seed fixes the 20 selected
examples; hosted generation itself is not guaranteed deterministic.

`max_tokens=131072` is an operational ceiling for **reasoning + answer**, not a requested
length or an SFT limit. It is not included in the prompt. A response that hits this ceiling
is flagged as incomplete; its raw text is saved in the journal, but is not placed in the
completed-output columns. No source rows are removed by the old 20,480-token SFT limit.

Changing the config or bundled generation code creates a separate run folder. Reuse the
same notebook/settings to resume. Hosted model aliases may change server-side: returned
model IDs, timestamps, fingerprints, usage, and response IDs are recorded per request.

In [ ]:
import yaml
CONFIG = yaml.safe_load((CODE_ROOT / "configs/regenerate_s1_deepseek.yaml").read_text())
# Optional edits go here, before creating the run in the next cell.
# CONFIG["api"]["concurrency"] = ...
print(yaml.safe_dump(CONFIG, sort_keys=False, allow_unicode=True))

In [ ]:
# @title Load the pinned Hugging Face source and prepare a resumable run
import importlib.metadata
from pipeline.regenerate_s1 import RegenerationRun, load_source, timestamp
from common.io import append_jsonl, read_jsonl
source_rows = load_source(CONFIG, HF_TOKEN)
run = RegenerationRun(source_rows, CONFIG, PROJECT_ROOT)
runtime_path = run.root / "runtime_sessions.jsonl"
read_jsonl(runtime_path, repair_tail=True)
append_jsonl(runtime_path, {"started_at": timestamp(), "python": sys.version,
    "packages": {name: importlib.metadata.version(name) for name in
                 ["datasets", "huggingface-hub", "httpx", "PyYAML", "tqdm"]}})
print("Source rows:", len(source_rows))
print("New columns:", run.reasoning_column, run.answer_column)
print("Smoke sample indices (seeded):", run.smoke_indices)
print("Drive output folder:", run.root)

## Smoke test — 20 random examples and a comparison CSV

This cell makes paid API requests for the 20 sampled rows that are not already complete.
The CSV has exactly these five columns, in this order:

`question`, `deepseek_thinking_trajectory`, `deepseek_attempt`,
`deepseek-v4-pro_reasoning`, `deepseek-v4-pro_answer`.

UTF-8 CSV preserves multiline text, quotation marks, and Unicode. Long reasoning can exceed
spreadsheet apps' cell-display limits; the CSV itself retains the full text. Blank generated
cells mean generation is incomplete/failed/pending or the final-response format is invalid;
inspect `smoke/status.jsonl` for details. Format checks require each of the four named
sections exactly once, in order, with nonempty content, followed by a single
`Final answer:` line and a nonempty conclusion. Markdown or plain headings are accepted.
Proofs can end with a textual conclusion; no box or length minimum is enforced.
These checks do not assess correctness or semantic quality. Review the answer column
using the checklist below; headings and words such as "check" are not quality evidence:

- Planning: is the method choice justified using this problem's facts or constraints?
- Solution and Evaluation: is the derivation complete, and do checks show their target,
  calculation or argument, and what the result establishes?
- Reflection: is a specific limitation identified and a concrete refinement shown,
  with an explanation of what improves? If the original approach is retained, is the
  tradeoff explained specifically rather than merely repeating that the solution is valid?
- Exploration: does an alternative contain a concrete equation, construction, or
  observation and a comparison with the main method? An unfinished alternative must
  not be presented as independent verification.

Assess reflection coverage separately from preservation of actual error corrections.
Compare claimed mistakes and corrections with the raw API reasoning column; a textual
match alone does not prove internal history. A smooth solution without corrections is
valid but still needs Reflection and Exploration. Raw returned text,
including format failures, remains in `attempts.jsonl`. Rerun to retry unsuccessful rows.
The old `deepseek_grade` is not a grade for the new answer, and no grading is run here.

In [ ]:
# @title Smoke test: generate the fixed random sample and save/download the CSV
try:
    await run.generate(run.smoke_indices, DEEPSEEK_API_KEY)
finally:
    smoke_result = run.export(run.smoke_indices, "smoke")
    print(smoke_result["summary"])
    print("CSV saved:", smoke_result["path"])
    print("Row status:", smoke_result["status_path"])
files.download(str(smoke_result["path"]))

## Full run — all 996 rows, preserving original columns

Set `RUN_FULL_GENERATION = True` below to run. There is no baseline-evaluation or smoke-completion gate.
Matching completed smoke results are reused automatically. After a runtime restart, run
setup/settings/source-loading, then this cell directly to resume.

The export is a Hugging Face-compatible **JSONL dataset** in original source order, with
every original field plus the two generated fields. Existing grades and responses retain
their original meaning; they are never overwritten. The source HF repository is unchanged.

`full/dataset.jsonl` is written when every row is complete. Otherwise the export is called
`full/dataset.partial.jsonl`, still containing every row, with null generated fields for
unsuccessful rows (including missing, duplicate, empty, or misordered required sections
and invalid final-answer markers). `full/status.jsonl`
lists failures by source index; `attempts.jsonl`
retains raw returned text, including incomplete responses and retry usage. Rerun the cell
to retry remaining rows. Authentication/balance/invalid-request errors stop the run and
save partial results. Failed responses are not silently promoted into the training data.

This notebook only regenerates data. It does not run SFT, evaluation, or HF publication.

In [ ]:
# @title Full run: generate all rows (completed smoke rows are reused)
RUN_FULL_GENERATION = False
if RUN_FULL_GENERATION:
    full_indices = list(range(len(source_rows)))
    try:
        await run.generate(full_indices, DEEPSEEK_API_KEY)
    finally:
        full_result = run.export(full_indices, "full")
        print(full_result["summary"])
        print("Dataset saved:", full_result["path"])
        print("Row status:", full_result["status_path"])
    if full_result["summary"]["remaining"]:
        print("Some rows remain incomplete. Inspect status.jsonl and rerun this cell.")
    else:
        print("All source rows now have generated reasoning and answers.")
else:
    print("Full run is off. Enable RUN_FULL_GENERATION to process all source rows.")

Load the finished export later with
`load_dataset("json", data_files=".../full/dataset.jsonl", split="train")`.
Drive stores the dataset and journals; save a notebook copy in Drive to retain cell output
displays too. The API journal records returned token usage, including retries. A request
whose response was lost or cancelled can still be billed; consult the provider dashboard.

References: [Kimi k1.5, §2.2](https://arxiv.org/html/2501.12599v1#S2.SS2),
[DeepSeek thinking fields](https://api-docs.deepseek.com/guides/thinking_mode/),
[DeepSeek API](https://api-docs.deepseek.com/api/create-chat-completion/).